# House Price Prediction - Model Training & Evaluation
**Person C: Model Training & Evaluation**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from house_price_predictor.data.loader import DataLoader
from house_price_predictor.data.preprocessor import DataPreprocessor
from house_price_predictor.models.trainer import ModelTrainer
from house_price_predictor.models.evaluator import ModelEvaluator
print('✓ Imports successful')

In [ ]:
# Load and preprocess data
loader = DataLoader(data_dir='data/raw')
train, test = loader.load_data()
combined = loader.combine_datasets()

preprocessor = DataPreprocessor(combined)
preprocessor.handle_missing_values()
preprocessor.remove_outliers()
preprocessor.encode_categorical()
preprocessor.transform_skewed_features()

train_data, val_data, test_data = preprocessor.create_train_validation_split()

In [ ]:
# Prepare features and target for training
X_train = train_data.drop('SalePrice', axis=1)
y_train = train_data['SalePrice']

X_val = val_data.drop('SalePrice', axis=1)
y_val = val_data['SalePrice']

print(f'✓ Training set: {X_train.shape}')
print(f'✓ Validation set: {X_val.shape}')

In [ ]:
# Train Random Forest model
# Using 100 trees as baseline - can experiment with different values
trainer = ModelTrainer(random_state=123)
model = trainer.train_random_forest(X_train, y_train, n_estimators=100)

In [ ]:
# Show which features matter most
importance_df = trainer.get_feature_importance(top_n=20)

In [ ]:
# Make predictions on both sets
train_pred = trainer.predict(X_train)
val_pred = trainer.predict(X_val)
print('✓ Predictions generated')

In [ ]:
# Evaluate model performance on training set
train_metrics = ModelEvaluator.evaluate_model(y_train, train_pred, name='Training Set')

In [ ]:
# Evaluate model performance on validation set
val_metrics = ModelEvaluator.evaluate_model(y_val, val_pred, name='Validation Set')

In [ ]:
# Visualize actual vs predicted prices on validation set
plt.figure(figsize=(10, 6))
plt.scatter(y_val, val_pred, alpha=0.5)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted Prices (Validation Set)')
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
plt.tight_layout()
plt.savefig('results/plots/actual_vs_predicted.png', dpi=100)
plt.show()
print('✓ Plot saved to results/plots/actual_vs_predicted.png')

In [ ]:
# Save the trained model
trainer.save_model('results/model.pkl')
print('✓ Model saved')

In [ ]:
# Make final predictions on test set
test_predictions = trainer.predict(test_data)

# Create submission file
submission = pd.DataFrame({
    'Id': range(1461, 1461 + len(test_predictions)),
    'SalePrice': test_predictions
})

submission.to_csv('results/predictions/submission.csv', index=False)
print(f'✓ Submission saved: {submission.shape[0]} predictions')
print(submission.head())